# Assignment 3

### Import data:
##### (Can take up to 10 min)

In [2]:
import sklearn.datasets as datasets


d_train = datasets.fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))

d_test = datasets.fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

##### Remove \n

In [3]:
import string
import re
from stop_words import get_stop_words

stop_words = set(get_stop_words('en'))
translator = str.maketrans('', '', string.punctuation)

data = []

for i in range(len(d_train.data)):
    text = d_train.data[i].replace('\n', ' ')
    text = text.translate(translator)
    text = text.split()
    text = [word for word in text if not re.search(r'\d', word)]
    text = [word for word in text if word.lower() not in stop_words]
    text = ' '.join(text).lower()
    data.append([d_train.target[i], text])

In [4]:
from collections import Counter
if(isinstance(data[0][1], str)):
    for i in range(len(data)):
        words = data[i][1].split()
        word_counts = Counter(words)
        data[i][1] = list(word_counts.items())

In [5]:
with open('keywords769.txt', 'r', encoding='utf-8') as f:
    keywords = [line.strip() for line in f if line.strip()]
keyword_set = set(keywords)
# Display the keyword_set contents (uses existing variable)
print(keyword_set)

{'pitching', 'different', 'fax', 'spiritual', 'standard', 'cable', 'blues', 'wings', 'military', 'history', 'motherboard', 'write', 'launch', 'therefore', 'space', 'program', 'yet', 'truth', 'churches', 'firearm', 'size', 'empire', 'nature', 'reason', 'mission', 'question', 'display', 'area', 'wolverine', 'xlib', 'controller', 'believe', 'wiretap', 'state', 'oil', 'waco', 'cubs', 'lot', 'goal', 'escrow', 'clinical', 'multiple', 'water', 'medical', 'chip', 'drives', 'know', 'published', 'come', 'played', 'newsletter', 'win', 'three', 'perhaps', 'village', 'look', 'processing', 'grace', 'speed', 'jobs', 'missions', 'bikes', 'mary', 'working', 'block', 'acts', 'cpu', 'things', 'small', 'problems', 'fans', 'book', 'soon', 'communications', 'drug', 'mail', 'manager', 'sale', 'includes', 'population', 'sell', 'clipper', 'players', 'bus', 'resources', 'tell', 'games', 'helmet', 'man', 'running', 'create', 'colormap', 'condition', 'eternal', 'even', 'saw', 'came', 'since', 'spacecraft', 'subje

### Model 1

In [6]:
y = []
samples = []
for i in range(len(data)):
    y.append(data[i][0])
    samples.append(data[i][1])

X_raw = []
for sample in samples:  # sample is list of (word, count) pairs
    d = dict(sample)
    d_lower = {k.lower(): v for k, v in d.items()}
    row = {kw: d_lower.get(kw, 0) for kw in keyword_set}
    X_raw.append(row)
from sklearn.feature_extraction import DictVectorizer

# Vectorize features
vectorizer = DictVectorizer(sparse=True)
X = vectorizer.fit_transform(X_raw)
from sklearn.linear_model import LogisticRegression

In [49]:
from sklearn.feature_extraction import DictVectorizer

# Vectorize features
vectorizer = DictVectorizer(sparse=True)
X = vectorizer.fit_transform(X_raw)

In [ ]:
# run the bigger grid search you prepared (this may take a while)
# use the GridSearchCV instance named `grid` defined earlier

# ensure X_train_texts and X_val_texts exist
try:
    X_train_texts, X_val_texts, y_train, y_val
except NameError:
    raw_texts = [' '.join([w for w, c in sample]) for sample in samples]
    X_train_texts, X_val_texts, y_train, y_val = train_test_split(
        raw_texts, y, test_size=0.20, stratify=y, random_state=42
    )

# ensure GridSearchCV instance exists (use existing pipeline/param_grid if present)
try:
    grid
except NameError:
    grid = GridSearchCV(pipeline, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)

grid.fit(X_train_texts, y_train)
print("Best CV score:", grid.best_score_)
print("Best params:", grid.best_params_)

# evaluate on validation split
y_val_pred = grid.predict(X_val_texts)
print("Validation accuracy:", accuracy_score(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred))

# retrain best estimator on train+val and evaluate official test set
best = grid.best_estimator_
best.fit(X_train_texts + X_val_texts, y_train + y_val)

# ensure y_test and raw_texts_test exist
try:
    y_test
except NameError:
    y_test = list(d_test.target)

try:
    raw_texts_test
except NameError:
    def preprocess_doc(doc):
        txt = doc.replace('\n', ' ')
        txt = txt.translate(translator)
        words = txt.split()
        words = [w for w in words if not re.search(r'\d', w)]
        words = [w.lower() for w in words if w.lower() not in stop_words]
        return ' '.join(words)
    raw_texts_test = [preprocess_doc(d) for d in d_test.data]

y_test_pred = best.predict(raw_texts_test)
print("Official test accuracy (best on train+val):", accuracy_score(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred))


IndentationError: expected an indented block after 'try' statement on line 5 (2980729924.py, line 6)